In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tqdm
import sys
import os
import pandas as pd
import time

In [3]:
from sentence_transformers import SentenceTransformer

encoding_model = SentenceTransformer("clip-ViT-B-32")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /home/kantz/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
import dotenv
import os

config = dotenv.dotenv_values(".env")
replace_keys = ["JAVA_HOME", "FUSEKI_HOME"]
append_keys = ["PATH"]
for key, value in config.items():
    # append to os.environ
    if key in append_keys:
        os.environ[key] = f"{os.environ.get(key, '')}:{value}"
    elif key in replace_keys:
        os.environ[key] = value

# check and compare values in fuseki log
os.environ["OPENBLAS_NUM_THREADS"] = "4"

In [5]:
sys.path.append("..")

In [6]:
from utils.datasets import DBPedia
from utils.dbs.qlever_native import QleverDBNative
from utils.dbs.fuseki_native import FusekiDBNative
from utils.datasets.base_dataset import QUERY_DIFFICULTY, QUERY_TYPE
from pathlib import Path
import numpy as np
from utils.datasets.base_dataset import DataTensor

2026-04-27 15:53:20,435 - INFO - Loading faiss with AVX512 support.
2026-04-27 15:53:20,439 - INFO - Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-04-27 15:53:20,488 - INFO - Loading faiss with AVX2 support.
2026-04-27 15:53:20,489 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-04-27 15:53:20,490 - INFO - Loading faiss.
2026-04-27 15:53:20,532 - INFO - Successfully loaded faiss.


In [7]:
dataset = DBPedia(base_dir=Path("../data/dbpedia"))

In [8]:
test_label = "horse on the bow"
test_tensor = DataTensor.from_numpy(encoding_model.encode(test_label))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
test_tensor.to_literal().n3()

'"{\\"data\\": [-0.04262268915772438, -0.1333884298801422, 0.08612467348575592, -0.02091798186302185, -0.15041583776474, 0.1737712323665619, -0.348517507314682, -0.9152146577835083, -0.3341490924358368, 0.17787908017635345, 0.05680551752448082, -0.20725640654563904, 0.5095284581184387, 0.18332251906394958, 0.11725624650716782, 0.1168459951877594, 0.20500661432743073, -0.1478302776813507, -0.007135884836316109, 0.4961417019367218, 0.3664361536502838, 0.41410648822784424, -0.04107498377561569, 0.5009220838546753, -0.2138804793357849, -0.23931393027305603, -0.7052134275436401, 0.4793880879878998, -0.17963360249996185, 0.2792432904243469, -0.06826944649219513, 0.0803503543138504, -0.20019735395908356, 0.25140535831451416, -0.16235408186912537, 0.1176009476184845, 0.10282524675130844, 0.20387926697731018, -0.04952261596918106, 0.26974374055862427, 0.13957971334457397, 0.5327335596084595, 0.12486020475625992, -0.08572806417942047, 0.1065823957324028, -0.06904290616512299, 0.19244082272052765

In [10]:
db_qlever = QleverDBNative(
    id="qlever",
    base_dir=Path(f"./scratch/{dataset.base_dir.name}"),
    dataset=dataset,
    use_encoded_ttl=True,
    name="QLever",
    port_offset=-100
)
db_no_tensor_idx = QleverDBNative(
    id="qlever",
    base_dir=Path(f"./scratch/{dataset.base_dir.name}"),
    dataset=dataset,
    use_encoded_ttl=True,
    enable_tensor_index=False,
    name="QLever (No Tensor Vocabulary)",
    port_offset=-200
)
db_fuseki = FusekiDBNative(
    id="fuseki",
    base_dir=Path(f"./scratch/{dataset.base_dir.name}"),
    dataset=dataset,
    use_encoded_ttl=True,
    name="Fuseki",
    exec_dir="../../jena-datatensor",
    port_offset=-300
)
possible_queries = db_qlever.get_queries(test_tensor)
print(possible_queries)
dbs: list[QleverDBNative | FusekiDBNative] = [db_qlever, db_no_tensor_idx, db_fuseki]
indices = ["index-encoded", "index-encoded-no-tidx", "fuseki-encoded"]
ids = ["dbpedia-encoded-tidx", "dbpedia-encoded", None]
for db, index, id in zip(dbs, indices, ids):
    db.db_dir = Path("../data/dbpedia") / index
    db.id = id if id is not None else db.id

2026-04-27 15:53:24,502 - WARNING - Killing any existing process using port 25943 before starting the server
2026-04-27 15:53:24,525 - ERROR - Command failed with return code 1
2026-04-27 15:53:24,526 - INFO - Initialized QLeverDBNative with id=qlever, port_id=25943, dataset=DBPedia, name=QLever, use_encoded_ttl=True, endpoint=http://localhost:25943/qlever-with-tidx/sparql
2026-04-27 15:53:24,527 - WARNING - Killing any existing process using port 25844 before starting the server
2026-04-27 15:53:24,547 - ERROR - Command failed with return code 1
2026-04-27 15:53:24,547 - INFO - Initialized QLeverDBNative with id=qlever, port_id=25844, dataset=DBPedia, name=QLever (No Tensor Vocabulary), use_encoded_ttl=True, endpoint=http://localhost:25844/qlever-no-tidx/sparql
2026-04-27 15:53:24,548 - WARNING - Killing any existing process using port 28745 before starting the server
2026-04-27 15:53:24,566 - ERROR - Command failed with return code 1


{<QUERY_DIFFICULTY.EASY: 'easy'>: {<QUERY_TYPE.EMBEDDED: 'embedded'>: '\nPREFIX dbr: <http://dbpedia.org/resource/>\nPREFIX dbo: <http://dbpedia.org/ontology/>\nPREFIX dtf: <https://w3id.org/rdf-tensor/functions#>\nSELECT DISTINCT ?s ?thumb_emb ?dist WHERE {\n    ?s a dbo:Ship ;\n         dbo:thumbnail_embedding ?thumb_emb .\n    BIND(dtf:dotProduct(?thumb_emb, "{\\"data\\": [-0.04262268915772438, -0.1333884298801422, 0.08612467348575592, -0.02091798186302185, -0.15041583776474, 0.1737712323665619, -0.348517507314682, -0.9152146577835083, -0.3341490924358368, 0.17787908017635345, 0.05680551752448082, -0.20725640654563904, 0.5095284581184387, 0.18332251906394958, 0.11725624650716782, 0.1168459951877594, 0.20500661432743073, -0.1478302776813507, -0.007135884836316109, 0.4961417019367218, 0.3664361536502838, 0.41410648822784424, -0.04107498377561569, 0.5009220838546753, -0.2138804793357849, -0.23931393027305603, -0.7052134275436401, 0.4793880879878998, -0.17963360249996185, 0.279243290424

In [11]:
triple_counts = []
for db in dbs[:-1]:
    with db:
        count = db.get_triple_count()
        count_tensors = db.query("""
PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT (COUNT(?v) AS ?count) WHERE {
    ?s dbo:thumbnail_embedding ?v .
}""")["count"].values[0]
        print(f"{db.name}: {count} triples, {count_tensors} tensors")
        triple_counts.append({
            "count": count,
            "tensors": count_tensors,
            "db": db.name
        })
print(f"Total triples in DB: {sum([tc['count'] for tc in triple_counts])}")

2026-04-27 15:53:24,679 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-27 15:53:24,680 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-27 15:53:24,681 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-27 15:53:24,682 - INFO - Stopping server!


2026-04-27 15:53:24,710 - ERROR - Command failed with return code 1
2026-04-27 15:53:24,711 - INFO - Starting QLever server on port 25943
2026-04-27 15:53:24,712 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-27 15:53:24,744 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:53:25,746 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:53:26,748 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:53:27,749 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:53:28,763 - I

QLever: 421850341 triples, 2369261 tensors


2026-04-27 15:53:31,038 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25844/qlever-no-tidx/sparql)
2026-04-27 15:53:32,040 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25844/qlever-no-tidx/sparql)
2026-04-27 15:53:33,052 - INFO - Server is up and responding to queries
2026-04-27 15:53:34,143 - INFO - Stopping server!
2026-04-27 15:53:34,213 - ERROR - Command failed with return code 1


QLever (No Tensor Vocabulary): 421850341 triples, 2369261 tensors
Total triples in DB: 843700682


In [12]:
counts_df = pd.DataFrame(triple_counts)
counts_df

,count,tensors,db
0,421850341,2369261,QLever
1,421850341,2369261,QLever (No Tensor Vocabulary)


In [13]:
counts_df["power"] = counts_df["count"].apply(lambda x: int(np.log10(x)))
counts_df["size"] = counts_df["count"]
counts_df["full_size"] = counts_df["count"]
counts_df["full_number_of_tensors"] = counts_df["tensors"]

In [14]:
from utils.helpers import pretty_print_counts

out_path_counts = Path("../scratch/results") / "dbpedia_counts.tex"
pretty_print_counts(counts_df, out_path_counts)

['power', 'size', 'full_size', 'full_number_of_tensors'] []


,Power,Generation $t$,$n$,$n_{tensors}$
0,8,421850341,421850341,2369261
1,8,421850341,421850341,2369261


In [15]:
with db_qlever as db:
    timings = []
    for _ in tqdm.tqdm(range(5)):
        start = time.time()
        r = db.query_auto(
            query_difficulty=QUERY_DIFFICULTY.EASY,
            query_type=QUERY_TYPE.INDEX,
            tensor=test_tensor,
        )
        end = time.time()
        timings.append(end - start)
print("Timings avg:", sum(timings) / len(timings))
r

2026-04-27 15:53:34,525 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-27 15:53:34,526 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-27 15:53:34,527 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-27 15:53:34,527 - INFO - Stopping server!
2026-04-27 15:53:34,594 - ERROR - Command failed with return code 1
2026-04-27 15:53:34,594 - INFO - Starting QLever server on port 25943
2026-04-27 15:53:34,595 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-27 15:53:34,597 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:53:35,598 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

Timings avg: 0.11848478317260742


,s,thumb_emb,dist
0,dbr:HMS_Warrior_(1860),"{""data"": [0.036129921674728394, 0.429935306310...",28.27962493896
1,dbr:La_Recouvrance_(schooner),"{""data"": [0.17521078884601593, -0.134947225451...",27.91348266602
2,dbr:Denis_Sullivan_(schooner),"{""data"": [0.07062532007694244, -0.230205625295...",27.86698150635
3,dbr:Falls_of_Clyde_(ship),"{""data"": [0.21303661167621613, -0.423036515712...",27.47668266296
4,dbr:SS_Great_Britain,"{""data"": [-0.03920137882232666, -0.19598063826...",27.4443397522
5,dbr:Elissa_(ship),"{""data"": [0.2183631807565689, 0.27063211798667...",27.43548774719
6,dbr:Skade_(yacht),"{""data"": [0.20522180199623108, -0.226250723004...",27.32021522522
7,dbr:L._A._Dunton_(schooner),"{""data"": [0.4466632306575775, -0.4638951420783...",27.07518768311
8,dbr:Pinta_(ship),"{""data"": [0.28884974122047424, -0.175495266914...",27.01076126099
9,dbr:SB_Mirosa,"{""data"": [-0.07320115715265274, -0.25511276721...",27.0029296875


In [16]:
with dbs[0] as db:
    res_native = db.query(f"""
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
SELECT * WHERE {{


    ?s a dbo:Ship .
    ?s dbo:thumbnail_embedding ?thumb_emb .
    ?s dbo:thumbnail_original ?thumb .      

    BIND(dtf:dotProduct(?thumb_emb, {test_tensor.to_literal().n3()}) AS ?dist)
}} 

ORDER BY DESC(?dist)
LIMIT 10""")
res_native

2026-04-27 15:53:39,359 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-27 15:53:39,360 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-27 15:53:39,361 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-27 15:53:39,361 - INFO - Stopping server!
2026-04-27 15:53:39,431 - ERROR - Command failed with return code 1
2026-04-27 15:53:39,432 - INFO - Starting QLever server on port 25943
2026-04-27 15:53:39,432 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-27 15:53:39,434 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:53:40,436 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

,s,thumb_emb,thumb,dist
0,dbr:HMS_Trent_(1757),"{""data"": [0.20562176406383514, 0.2037418782711...",https://upload.wikimedia.org/wikipedia/commons...,31.31344604492
1,dbr:Fram,"{""data"": [0.2554498314857483, -0.1410300731658...",https://upload.wikimedia.org/wikipedia/commons...,29.51530838013
2,dbr:Fram,"{""data"": [0.2554498314857483, -0.1410300731658...",https://upload.wikimedia.org/wikipedia/commons...,29.51530838013
3,dbr:Fram,"{""data"": [0.2554498314857483, -0.1410300731658...",http://upload.wikimedia.org/wikipedia/commons/...,29.51530838013
4,dbr:Fram,"{""data"": [0.2554498314857483, -0.1410300731658...",https://upload.wikimedia.org/wikipedia/commons...,29.51530838013
5,dbr:Fram,"{""data"": [0.2554498314857483, -0.1410300731658...",https://upload.wikimedia.org/wikipedia/commons...,29.51530838013
6,dbr:Fram,"{""data"": [0.2554498314857483, -0.1410300731658...",http://upload.wikimedia.org/wikipedia/commons/...,29.51530838013
7,dbr:USCGC_Eagle_(WIX-327),"{""data"": [0.288393497467041, 0.597437918186187...",https://upload.wikimedia.org/wikipedia/commons...,29.12233352661
8,dbr:USCGC_Eagle_(WIX-327),"{""data"": [0.288393497467041, 0.597437918186187...",http://upload.wikimedia.org/wikipedia/commons/...,29.12233352661
9,dbr:USCGC_Eagle_(WIX-327),"{""data"": [0.288393497467041, 0.597437918186187...",http://upload.wikimedia.org/wikipedia/commons/...,29.12233352661


In [17]:
with dbs[0] as db:
    res =db.query(f"""
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
SELECT * WHERE {{
SERVICE tensorIndex: {{
    _:config tensorIndex:numNN 10 ;
    tensorIndex:left ?query_vector ;
    tensorIndex:bindDistance ?dist ;
    tensorIndex:payload ?s, ?thumb ;
    tensorIndex:searchK 64 ;
    tensorIndex:kIVF 128 ;
    # tensorIndex:experimentalRightCacheName "easy_index_dbpedia" ;
    tensorIndex:right ?thumb_emb ;
    tensorIndex:algorithm tensorIndex:ivf ;
    tensorIndex:distance tensorIndex:dot .
       {{
            ?s a dbo:Ship ;
            dbo:thumbnail_embedding ?thumb_emb ;
            dbo:thumbnail_original ?thumb .
        }}
    }}
    VALUES (?query_vector) {{ ({test_tensor.to_literal().n3()}) }}
}} 
ORDER BY DESC(?dist)
""")
res

2026-04-27 15:53:43,853 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-27 15:53:43,854 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-27 15:53:43,854 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-27 15:53:43,855 - INFO - Stopping server!
2026-04-27 15:53:43,922 - ERROR - Command failed with return code 1
2026-04-27 15:53:43,923 - INFO - Starting QLever server on port 25943
2026-04-27 15:53:43,923 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-27 15:53:43,925 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:53:44,926 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

,query_vector,dist,s,thumb,thumb_emb
0,"{""data"": [-0.04262268915772438, -0.13338842988...",27.4443397522,dbr:SS_Great_Britain,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [-0.03920137882232666, -0.19598063826..."
1,"{""data"": [-0.04262268915772438, -0.13338842988...",27.4443397522,dbr:SS_Great_Britain,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [-0.03920137882232666, -0.19598063826..."
2,"{""data"": [-0.04262268915772438, -0.13338842988...",27.4443397522,dbr:SS_Great_Britain,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [-0.03920137882232666, -0.19598063826..."
3,"{""data"": [-0.04262268915772438, -0.13338842988...",27.4443397522,dbr:SS_Great_Britain,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [-0.03920137882232666, -0.19598063826..."
4,"{""data"": [-0.04262268915772438, -0.13338842988...",27.4443397522,dbr:SS_Great_Britain,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [-0.03920137882232666, -0.19598063826..."
5,"{""data"": [-0.04262268915772438, -0.13338842988...",27.4443397522,dbr:SS_Great_Britain,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [-0.03920137882232666, -0.19598063826..."
6,"{""data"": [-0.04262268915772438, -0.13338842988...",27.4443397522,dbr:SS_Great_Britain,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [-0.03920137882232666, -0.19598063826..."
7,"{""data"": [-0.04262268915772438, -0.13338842988...",27.4443397522,dbr:SS_Great_Britain,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [-0.03920137882232666, -0.19598063826..."
8,"{""data"": [-0.04262268915772438, -0.13338842988...",27.4443397522,dbr:SS_Great_Britain,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [-0.03920137882232666, -0.19598063826..."
9,"{""data"": [-0.04262268915772438, -0.13338842988...",27.4443397522,dbr:SS_Great_Britain,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [-0.03920137882232666, -0.19598063826..."


In [18]:
res.to_csv(Path("scratch") / "dbpedia_results_index.csv", index=False)

In [19]:
with dbs[1] as db:
    db.query_auto(test_tensor, query_type=QUERY_TYPE.INDEX, query_difficulty=QUERY_DIFFICULTY.HARD)

2026-04-27 15:53:48,071 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-no-tidx/qlever-no-tidx_run.log
2026-04-27 15:53:48,072 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-27 15:53:48,073 - WARNING - DB directory ../data/dbpedia/index-encoded-no-tidx already exists!
2026-04-27 15:53:48,073 - INFO - Stopping server!
2026-04-27 15:53:48,140 - ERROR - Command failed with return code 1
2026-04-27 15:53:48,140 - INFO - Starting QLever server on port 25844
2026-04-27 15:53:48,141 - INFO - Running command: qlever-server -i dbpedia-encoded --port 25844 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-27 15:53:48,142 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25844/qlever-no-tidx/sparql)
2026-04-27 15:53:49,144 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25844/qlever-no-tidx/sparql)
2

In [20]:
from utils.helpers import ndcgscore_query
reference_result = res_native
score = ndcgscore_query(res, reference_result, k=10)
print(f"Score: {score}")

Score: 0.8042472959280005


In [21]:
# count ships


with dbs[0] as db:
    rs =db.query(f"""
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT (COUNT(*) AS ?count) WHERE {{ 
            ?s a dbo:Ship ;
            dbo:thumbnail_embedding ?thumb_emb .
}} LIMIT 10""")
    rr =  db.query(f"""PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT (COUNT(*) AS ?count) WHERE {{ 
            ?s a dbo:RailwayLine ;
            dbo:thumbnail_embedding ?thumb_emb .
}} LIMIT 10""")
rs.iloc[0], rr.iloc[0]

2026-04-27 15:53:59,738 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-27 15:53:59,739 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-27 15:53:59,740 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-27 15:53:59,740 - INFO - Stopping server!
2026-04-27 15:53:59,810 - ERROR - Command failed with return code 1
2026-04-27 15:53:59,811 - INFO - Starting QLever server on port 25943
2026-04-27 15:53:59,812 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-27 15:53:59,814 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:54:00,815 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

(count    25031
 Name: 0, dtype: str,
 count    7548
 Name: 0, dtype: str)

In [22]:
# for find similar thumbnails between ships and railroads
times = []
with dbs[0] as db:
    #     warmup = db.query("""
    # PREFIX dbr: <http://dbpedia.org/resource/>
    # PREFIX dbo: <http://dbpedia.org/ontology/>
    # PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
    # SELECT * WHERE {
    #     { SELECT * WHERE {
    #         ?r a dbo:Work .
    #         ?r dbo:thumbnail_embedding ?thumb_rail_emb .
    #         ?r dbo:thumbnail_original ?thumb_rail .
    #     } LIMIT 1
    #     }
    #     SERVICE tensorIndex: {
    #     _:config tensorIndex:numNN 10 ;
    #     tensorIndex:left ?thumb_rail_emb ;
    #     tensorIndex:bindDistance ?dist ;
    #     tensorIndex:payload ?s, ?thumb_ship ;
    #     tensorIndex:searchK 1 ;
    #     tensorIndex:nTrees 128 ;
    #     tensorIndex:experimentalRightCacheName "hard_index_dbpedia" ;
    #     tensorIndex:right ?thumb_emb_ship ;
    #     tensorIndex:algorithm tensorIndex:ivf ;
    #     tensorIndex:distance tensorIndex:dot .
    #         {
    #             ?s a dbo:Ship ;
    #             dbo:thumbnail_embedding ?thumb_emb_ship ;
    #             dbo:thumbnail_original ?thumb_ship .
    #         }
    #     }
    #     }""")
    for _ in range(0):
        start = time.time()

        noised_tensor = DataTensor.from_numpy(
            test_tensor.data + np.random.normal(scale=0.001, size=test_tensor.shape)
        )
        railway_ship_assoc = db.query(f"""
    PREFIX dbr: <http://dbpedia.org/resource/>
    PREFIX dbo: <http://dbpedia.org/ontology/>
    PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
    PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
    SELECT DISTINCT ?r ?s  ?dist ?thumb_rail_emb ?thumb_ship_emb WHERE {{
        {{
        SELECT DISTINCT ?r ?s ?dist ?thumb_rail_emb ?thumb_ship_emb WHERE {{
            
            ?r a dbo:RailwayLine .
            ?r dbo:thumbnail_embedding ?thumb_rail_emb .
                                        
            SERVICE tensorIndex: {{
            _:config tensorIndex:numNN 1 ;
            tensorIndex:left ?thumb_rail_emb ;
            tensorIndex:bindDistance ?dist ;
            tensorIndex:payload ?s, ?thumb_ship ;
            tensorIndex:searchK 1 ;
            tensorIndex:nTrees 512 ;
            tensorIndex:experimentalRightCacheName "hard_index_dbpedia" ;
            tensorIndex:right ?thumb_ship_emb ;
            tensorIndex:algorithm tensorIndex:ivf ;
            tensorIndex:distance tensorIndex:dot .
            {{
                ?s a dbo:Ship ;
                dbo:thumbnail_embedding ?thumb_ship_emb ;    
            }}
            }}                  
        }}
        }}
        VALUES (?some_emb) {{ ({noised_tensor.to_literal().n3()}) }}
                                    
    }}
    ORDER BY DESC(?dist)
    LIMIT 20""")
        end = time.time()
        delta = end - start
        times.append(delta)
times

2026-04-27 15:54:04,042 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-27 15:54:04,043 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-27 15:54:04,044 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-27 15:54:04,045 - INFO - Stopping server!
2026-04-27 15:54:04,112 - ERROR - Command failed with return code 1
2026-04-27 15:54:04,113 - INFO - Starting QLever server on port 25943
2026-04-27 15:54:04,113 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-27 15:54:04,115 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:54:05,117 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

[]

In [23]:
with db_qlever as db:
    hard_results = db.query(
        f"""
        PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
SELECT DISTINCT ?r ?s  ?dist ?thumb_work_emb ?thumb_settlement_emb WHERE {{
    {{
    SELECT DISTINCT ?r ?s ?dist ?thumb_work_emb ?thumb_settlement_emb WHERE {{
        
        ?r a dbo:RailwayLine;
         dbo:thumbnail_embedding ?thumb_work_emb ;
         dbo:thumbnail_original ?thumb_work_original .
        BIND(LCASE(STR(?thumb_work_original)) AS ?thumb_work)
        FILTER(STRENDS(?thumb_work, ".png") || STRENDS(?thumb_work, ".jpg") || STRENDS(?thumb_work, ".jpeg")) .
                                    
        SERVICE tensorIndex: {{
        _:config tensorIndex:numNN 1 ;
        tensorIndex:left ?thumb_work_emb ;
        tensorIndex:bindDistance ?dist ;
        tensorIndex:payload ?s, ?thumb_settlement_emb ;
        tensorIndex:searchK 1 ;
        tensorIndex:kIVF 128 ;
        tensorIndex:experimentalRightCacheName "hard_index_dbpedia" ;
        tensorIndex:right ?thumb_settlement_emb ;
        tensorIndex:algorithm tensorIndex:ivf ;
        tensorIndex:distance tensorIndex:dot .
        {{
            ?s a dbo:Ship ;
            dbo:thumbnail_embedding ?thumb_settlement_emb ;  
            dbo:thumbnail_original ?thumb_settlement_original .
            BIND(LCASE(STR(?thumb_settlement_original)) AS ?thumb_settlement)
            FILTER(STRENDS(?thumb_settlement, ".png") || STRENDS(?thumb_settlement, ".jpg") || STRENDS(?thumb_settlement, ".jpeg")) .
        }}
        }}                  
    }}
    }}
    VALUES (?some_emb) {{ ({test_tensor.to_literal().n3()}) }}         
}}
ORDER BY DESC(?dist)
LIMIT 10"""
    )
    # hard_results = db.query_auto(
    #     test_tensor, query_type=QUERY_TYPE.INDEX, query_difficulty=QUERY_DIFFICULTY.HARD
    # )
    easy_results = db.query_auto(
        test_tensor, query_type=QUERY_TYPE.INDEX, query_difficulty=QUERY_DIFFICULTY.EASY
    )

2026-04-27 15:54:07,284 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-27 15:54:07,285 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-27 15:54:07,285 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-27 15:54:07,286 - INFO - Stopping server!
2026-04-27 15:54:07,355 - ERROR - Command failed with return code 1
2026-04-27 15:54:07,356 - INFO - Starting QLever server on port 25943
2026-04-27 15:54:07,356 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-27 15:54:07,358 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:54:08,359 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

In [24]:
easy_results.to_csv(Path("scratch") / "dbpedia_easy_results.csv", index=False)
easy_results

,s,thumb_emb,dist
0,dbr:HMS_Warrior_(1860),"{""data"": [0.036129921674728394, 0.429935306310...",28.27962493896
1,dbr:La_Recouvrance_(schooner),"{""data"": [0.17521078884601593, -0.134947225451...",27.91348266602
2,dbr:Denis_Sullivan_(schooner),"{""data"": [0.07062532007694244, -0.230205625295...",27.86698150635
3,dbr:Falls_of_Clyde_(ship),"{""data"": [0.21303661167621613, -0.423036515712...",27.47668266296
4,dbr:SS_Great_Britain,"{""data"": [-0.03920137882232666, -0.19598063826...",27.4443397522
5,dbr:Elissa_(ship),"{""data"": [0.2183631807565689, 0.27063211798667...",27.43548774719
6,dbr:Skade_(yacht),"{""data"": [0.20522180199623108, -0.226250723004...",27.32021522522
7,dbr:L._A._Dunton_(schooner),"{""data"": [0.4466632306575775, -0.4638951420783...",27.07518768311
8,dbr:Pinta_(ship),"{""data"": [0.28884974122047424, -0.175495266914...",27.01076126099
9,dbr:SB_Mirosa,"{""data"": [-0.07320115715265274, -0.25511276721...",27.0029296875


In [25]:
easy_results.loc[0, "s"], easy_results.loc[0, "thumb_emb"], easy_results.loc[0, "dist"]

('dbr:HMS_Warrior_(1860)',
 '{"data": [0.036129921674728394, 0.4299353063106537, -0.37877365946769714, 0.30682092905044556, 0.322639524936676, 0.2249593734741211, -0.06521791219711304, 0.17391973733901978, -0.2688770294189453, 0.38226354122161865, 0.152025043964386, -0.19757014513015747, 0.7534399032592773, 0.758370578289032, -0.21658317744731903, -0.02999865636229515, 0.23736977577209473, 0.1001720055937767, 0.42510396242141724, 0.14974050223827362, 0.0637899860739708, 0.11872407793998718, -0.35933512449264526, 0.2847185730934143, 0.43100637197494507, 0.46448808908462524, -0.29308536648750305, 0.39990168809890747, -0.04956104978919029, 0.07813140004873276, -0.5828685164451599, 0.1557556390762329, 0.06378577649593353, 0.5864004492759705, 0.1866665631532669, 0.0543186292052269, -0.04140060395002365, -0.18815915286540985, -0.2874862551689148, 1.5594831705093384, 0.19858968257904053, -0.021679505705833435, 0.38822221755981445, 0.5762841105461121, 0.28483134508132935, -0.5699039697647095, 

In [26]:
hard_results

,r,s,dist,thumb_work_emb,thumb_settlement_emb
0,dbr:Chengdu_Metro,dbr:Virginia-class_submarine,149.7560424805,"{""data"": [0.22622768580913544, 0.0546032562851...","{""data"": [-0.1000426858663559, -0.080208711326..."
1,dbr:Taipa_line,dbr:Virginia-class_submarine,148.5401763916,"{""data"": [0.03102342039346695, -0.059559382498...","{""data"": [-0.1000426858663559, -0.080208711326..."
2,dbr:Yangluo_Line,dbr:Virginia-class_submarine,146.5701904297,"{""data"": [0.13836224377155304, 0.1058328151702...","{""data"": [-0.1000426858663559, -0.080208711326..."
3,dbr:Buenos_Aires_Underground,dbr:Virginia-class_submarine,140.4429473877,"{""data"": [-0.057316623628139496, -0.1589123308...","{""data"": [-0.1000426858663559, -0.080208711326..."
4,dbr:Line_13_(CPTM),dbr:Virginia-class_submarine,139.5857849121,"{""data"": [-0.21847796440124512, -0.04886087775...","{""data"": [-0.1000426858663559, -0.080208711326..."
5,dbr:Orlyval,dbr:Virginia-class_submarine,138.4967041016,"{""data"": [0.11910116672515869, -0.394591093063...","{""data"": [-0.1000426858663559, -0.080208711326..."
6,dbr:Abbey_Line,dbr:Virginia-class_submarine,137.7716217041,"{""data"": [0.17977674305438995, -0.300230383872...","{""data"": [-0.1000426858663559, -0.080208711326..."
7,dbr:Kolkata_Metro,dbr:Virginia-class_submarine,137.671875,"{""data"": [-0.012433663010597229, -0.1900815814...","{""data"": [-0.1000426858663559, -0.080208711326..."
8,dbr:London_Overground,dbr:Virginia-class_submarine,137.6464233398,"{""data"": [0.02959146350622177, 0.0380432158708...","{""data"": [-0.1000426858663559, -0.080208711326..."
9,dbr:CDGVAL,dbr:Virginia-class_submarine,135.4984130859,"{""data"": [-0.17296558618545532, 0.003928374499...","{""data"": [-0.1000426858663559, -0.080208711326..."


In [27]:
# load images for both results
from utils.dbs.base_db import BaseDB


def enhance_col_with_thumbs(
    db: BaseDB, df: pd.DataFrame, dbr_col: str, thumb_col_name="thumb"
) -> pd.DataFrame:

    thumbs = []
    g = tqdm.tqdm(df[dbr_col], desc=f"Fetching thumbnails for {thumb_col_name}")
    for s in g:
        g.set_description(f"Fetching thumbnails for {thumb_col_name}: '{s}'")
        s = f"<{s.replace('dbr:', 'http://dbpedia.org/resource/')}>"
        res = db.query(f"""PREFIX dbr: <http://dbpedia.org/resource/>
                            PREFIX dbo: <http://dbpedia.org/ontology/>
                            SELECT ?thumb ?thumb_lc WHERE {{
                                {s} dbo:thumbnail_original ?thumb .
                                BIND (LCASE(STR(?thumb)) AS ?thumb_lc) 
                                FILTER (STRENDS(?thumb_lc, ".png") || STRENDS(?thumb_lc, ".jpg") || STRENDS(?thumb_lc, ".jpeg"))
                            }} LIMIT 1""")
        # print(f"Query result for {s}: {res}")
        if len(res) > 0:
            thumbs.append(res["thumb"].values[0])
        else:
            thumbs.append(None)
    df[thumb_col_name] = thumbs
    return df


with db_qlever as db:
    easy_results = enhance_col_with_thumbs(db, easy_results, "s")
    hard_results = enhance_col_with_thumbs(
        db, hard_results, "s", thumb_col_name="thumb_s"
    )
    hard_results = enhance_col_with_thumbs(
        db, hard_results, "r", thumb_col_name="thumb_r"
    )



2026-04-27 15:54:17,241 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-27 15:54:17,242 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-27 15:54:17,243 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-27 15:54:17,244 - INFO - Stopping server!
2026-04-27 15:54:17,314 - ERROR - Command failed with return code 1
2026-04-27 15:54:17,315 - INFO - Starting QLever server on port 25943
2026-04-27 15:54:17,315 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-27 15:54:17,318 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:54:18,319 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

In [28]:
hard_results.to_csv(Path("scratch") / "dbpedia_hard_results.csv", index=False)
easy_results.to_csv(Path("scratch") / "dbpedia_easy_results.csv", index=False)

In [29]:
easy_results.loc[0, 'thumb'].split('/')[-1]

'HMS_Warrior_Pembroke_Dock_July_1977_B.jpg'

## Token

You need to set your Oauth tokens in the local `.env`:
```sh
MEDIA_WIKI_TOKEN="..."
MEDIA_WIKI_SECRET="..."
MEDIA_WIKI_ACCESS_TOKEN="..."
MEDIA_WIKI_ACCESS_SECRET="..."
```
You can register a new token [here](https://meta.wikimedia.org/wiki/Special:OAuthConsumerRegistration/propose/oauth1a).

In [30]:
import requests
from requests_oauthlib import OAuth1
import dotenv
from utils.datasets.dbpedia_utils.helpers import USER_AGENT

dotenv.load_dotenv()  # to load MEDIAWIKI_TOKEN from .env file

auth = OAuth1(
    os.getenv("MEDIA_WIKI_TOKEN"),
    os.getenv("MEDIA_WIKI_SECRET"),
    os.getenv("MEDIA_WIKI_ACCESS_TOKEN"),
    os.getenv("MEDIA_WIKI_ACCESS_SECRET"),
)   

In [31]:
auth.client

<Client client_key=eea9f51d9008d744c272c3ece851b184, client_secret=****, resource_owner_key=914b284deef1f1f0525ea803d55914b0, resource_owner_secret=****, signature_method=HMAC-SHA1, signature_type=AUTH_HEADER, callback_uri=None, rsa_key=None, verifier=None, realm=None, encoding=utf-8, decoding=utf-8, nonce=None, timestamp=None>

In [32]:
url=f"https://commons.wikimedia.org/wiki/Special:FilePath/{easy_results.loc[0, 'thumb'].split('/')[-1]}?width=330"
resp = requests.get(
    url,
    auth=auth,
    headers={"User-Agent": USER_AGENT},
)

In [33]:
import pywikibot

pywikibot.config.usernames['commons']['commons'] = "Dakantz"
# set user-agent

authenticate = (
    os.getenv("MEDIA_WIKI_TOKEN"),
    os.getenv("MEDIA_WIKI_SECRET"),
    os.getenv("MEDIA_WIKI_ACCESS_TOKEN"),
    os.getenv("MEDIA_WIKI_ACCESS_SECRET"),
)
pywikibot.config.authenticate['commons.wikimedia.org'] = authenticate
pywikibot.config.user_agent = USER_AGENT
site = pywikibot.Site('commons', 'commons')
site.login()

In [34]:
site.allimages()

In [35]:
from PIL import Image
fname = easy_results.loc[0, "thumb"].split("/")[-1]
img = pywikibot.FilePage(site, fname)

img.download(filename=fname, url_width=330)
img = Image.open(fname)

In [36]:
from utils.datasets.dbpedia_utils import image_formatter


def thumbs_to_pil(
    thumbs: pd.Series, scratch_dir=Path("../scratch/dbpedia_thumbs")
) -> pd.Series:

    scratch_dir.mkdir(parents=True, exist_ok=True)
    pil_images = []
    g = tqdm.tqdm(thumbs, desc="Fetching thumbnails")
    for thumb in g:
        g.set_description(f"Fetching thumbnail for {thumb}...")
        # url_thumb = get_wc_thumb(thumb)

        fname = thumb.split("/")[-1]
        out_f = scratch_dir / fname
        if not out_f.exists():
            img = pywikibot.FilePage(site, fname)
            img.download(filename=out_f, url_width=330)
        img = Image.open(out_f)
        pil_images.append(img)
    return pd.Series(pil_images, index=thumbs.index)


easy_results["thumb_img"] = thumbs_to_pil(easy_results["thumb"])
hard_results["thumb_s_img"] = thumbs_to_pil(hard_results["thumb_s"])
hard_results["thumb_r_img"] = thumbs_to_pil(hard_results["thumb_r"])

Fetching thumbnail for https://upload.wikimedia.org/wikipedia/commons/2/23/HMS_Warrior_Pembroke_Dock_July_1977_B.jpg...:   0%|          | 0/10 [00:00<?, ?it/s]

Fetching thumbnail for http://upload.wikimedia.org/wikipedia/commons/0/0d/SS_Great_Britain_diagram.jpg...:  40%|████      | 4/10 [00:08<00:12,  2.15s/it]              WARNING: Http response status 429
Fetching thumbnail for http://upload.wikimedia.org/wikipedia/commons/0/0d/SS_Great_Britain_diagram.jpg...:  40%|████      | 4/10 [00:11<00:16,  2.79s/it]


FileNotFoundError: [Errno 2] No such file or directory: '../scratch/dbpedia_thumbs/SS_Great_Britain_diagram.jpg'

In [ ]:
out_dir = Path("../scratch/results/")


def to_tex_with_thumbs(
    df,
    t_cols=["thumb_img"],
    col_mapping={
        "s": "Ship",
        "r": "Railway",
        "thumb_img_tex": "Thumbnail",
        "dist": "Distance",
    },
    base_dir="figures/generated",
    sub_dir="dbpedia/media",
    out_name="dbpedia_easy_results.tex",
    col_format ="lp{3cm}r"
):
    out_dir.mkdir(parents=True, exist_ok=True)
    results_dir = Path(out_dir) / sub_dir
    results_dir.mkdir(parents=True, exist_ok=True)
    fig_dir = Path(base_dir) / sub_dir
    for t_col in t_cols:
        df[f"{t_col}_tex"] = df[t_col]
        
        for i, r in df.iterrows():
            if r[t_col] is not None and isinstance(r[t_col], Image.Image):
                fname = f"{t_col}_{i}.png"
                fig_file = fig_dir / fname
                im_file = results_dir / fname
                df.at[i, f"{t_col}_tex"] = f"\\includegraphics[width=3cm]{{{fig_file}}}"
                print(f"Saving image for row {i} to {im_file} and referencing as {fig_file} in LaTeX", r[t_col])
                img: Image = r[t_col]
                img.save(im_file)
            else:
                df.at[i, f"{t_col}_tex"] = "No image"
    df_renamed = df.rename(columns=col_mapping)
    print(f"Renamed columns for LaTeX: {df_renamed.columns}")
    allowed_cols = [c for c in list(col_mapping.values()) if c in df_renamed.columns]
    print(f"Allowed columns for LaTeX output: {allowed_cols}")
    df_renamed = df_renamed[allowed_cols]
    df_renamed.set_index(allowed_cols[0], inplace=True)
    df_renamed.style.to_latex(
        buf= results_dir / out_name,
        column_format=col_format,
    )


to_tex_with_thumbs(easy_results)

Saving image for row 0 to ../scratch/results/dbpedia/media/thumb_img_0.png and referencing as figures/generated/dbpedia/media/thumb_img_0.png in LaTeX <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=330x247 at 0x7FF50012B5C0>
Saving image for row 1 to ../scratch/results/dbpedia/media/thumb_img_1.png and referencing as figures/generated/dbpedia/media/thumb_img_1.png in LaTeX <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=330x217 at 0x7FF500128050>
Saving image for row 2 to ../scratch/results/dbpedia/media/thumb_img_2.png and referencing as figures/generated/dbpedia/media/thumb_img_2.png in LaTeX <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=330x250 at 0x7FF4FBFE3360>
Renamed columns for LaTeX: Index(['Ship', 'thumb_emb', 'Distance', 'thumb', 'thumb_img', 'Thumbnail'], dtype='str')
Allowed columns for LaTeX output: ['Ship', 'Thumbnail', 'Distance']
